# 01 Beta XGB1 Day-Candidate Diagnostics

This diagnostic notebook investigates why `m8_xgb` XGB1 day-candidate performance transfers poorly from Alpha to Beta.

**Purpose.** Diagnose model behaviour only. This notebook does not train models, tune thresholds, normalize features, or modify the main journal workflow.

**Inputs.** Existing real-run Notebook 2 `m8_xgb` prediction CSVs from `outputs/intermediate/02_correction_validation/`.

**Outputs.** Local diagnostic CSVs, PNG figures, and Plotly HTML example browsers under `notebooks/99_Misc/outputs/01_beta_xgb1_diagnostics/`.

**Key idea.** The current prediction CSVs do not store XGB1 day probabilities directly, but they do store `prob_interval` on days that XGB1 passed into XGB2. Therefore this notebook infers `xgb1_candidate_day = any(prob_interval.notna())` at site-day level.


## 1. Imports, Paths, And Run Guard

This section resolves the article root, creates local diagnostic output folders, and checks that Notebook 2 has already produced real model outputs. If the correction-validation manifest still says placeholder mode, this notebook stops early with a clear error.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

# Plotting is diagnostic-only and writes local PNG/HTML artifacts.
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Resolve publication/2_journal_article robustly from VS Code, JupyterLab, or repo root.
start = Path.cwd().resolve()
article_root = None
for candidate in [start, *start.parents]:
    if candidate.name == "2_journal_article" and (candidate / "dataset").exists():
        article_root = candidate
        break
    nested = candidate / "publication" / "2_journal_article"
    if (nested / "dataset").exists():
        article_root = nested.resolve()
        break
if article_root is None:
    raise RuntimeError(f"Could not locate publication/2_journal_article from {start}")

prediction_dir = article_root / "outputs" / "intermediate" / "02_correction_validation"
manifest_path = article_root / "outputs" / "manifests" / "02_correction_validation.json"
notebook_dir = article_root / "notebooks" / "99_Misc"
output_dir = notebook_dir / "outputs" / "01_beta_xgb1_diagnostics"
csv_dir = output_dir / "csv"
fig_dir = output_dir / "figures"
html_dir = output_dir / "html_examples"
for directory in [csv_dir, fig_dir, html_dir]:
    directory.mkdir(parents=True, exist_ok=True)

if not manifest_path.exists():
    raise FileNotFoundError(
        "Notebook 2 manifest is missing. Run 02_correction_validation.ipynb in full mode first."
    )
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("status") != "complete" or manifest.get("contains_placeholder_metrics"):
    raise RuntimeError(
        "Notebook 2 outputs are not real full-run metrics. "
        f"Manifest status={manifest.get('status')!r}, "
        f"contains_placeholder_metrics={manifest.get('contains_placeholder_metrics')!r}."
    )

required_prediction_files = {
    "Alpha alpha_F": prediction_dir / "02_correction_predictions_alpha_holdout_alpha_F_m8_xgb.csv",
    "Alpha alpha_E": prediction_dir / "04_correction_predictions_alpha_holdout_alpha_E_m8_xgb.csv",
    "Alpha alpha_G": prediction_dir / "06_correction_predictions_alpha_holdout_alpha_G_m8_xgb.csv",
    "Beta transfer": prediction_dir / "08_correction_predictions_beta_transfer_m8_xgb.csv",
}
missing = [str(path) for path in required_prediction_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required m8_xgb prediction files:\n" + "\n".join(missing))

print(f"Article root: {article_root}")
print(f"Prediction dir: {prediction_dir}")
print(f"Diagnostic output dir: {output_dir}")


## 2. Helper Functions

These helpers parse the existing prediction CSVs, aggregate interval predictions into site-day diagnostics, compute confusion metrics, and select deterministic example days. The notebook keeps XGB1 candidate-day performance separate from final day performance after XGB2 interval thresholding.


In [ ]:
EXPECTED_COLUMNS = [
    "substation_id",
    "date",
    "timestamp",
    "net_load_MW",
    "solar_MW",
    "label_interval",
    "label_day",
    "pred_interval",
    "corrected_net_load_MW",
    "prob_interval",
]
CONFUSION_ORDER = ["TP", "FP", "FN", "TN"]
SCORE_COLUMNS = ["precision", "recall", "f1"]
PRIORITY_BETA_SITES = ["beta_B", "beta_G", "beta_F", "beta_D"]
PALETTE = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
    "label": "rgba(235, 147, 44, 0.18)",
    "pred": "rgba(34, 48, 61, 0.16)",
}


def parse_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.astype("string").str.strip().str.lower().isin(["true", "1", "yes", "y"])


def parse_wall_clock(series: pd.Series) -> pd.Series:
    text = series.astype("string").str.strip()
    stripped = text.str.replace(r"(Z|[+-]\d{2}:\d{2})$", "", regex=True)
    return pd.to_datetime(stripped, errors="raise")


def load_prediction_csv(path: Path, dataset: str, fold_id: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing = [col for col in EXPECTED_COLUMNS if col not in frame.columns]
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")
    frame = frame[EXPECTED_COLUMNS].copy()
    frame["dataset"] = dataset
    frame["fold_id"] = fold_id
    frame["date"] = frame["date"].astype(str)
    frame["timestamp_dt"] = parse_wall_clock(frame["timestamp"])
    frame["label_interval"] = parse_bool(frame["label_interval"])
    frame["label_day"] = parse_bool(frame["label_day"])
    frame["pred_interval"] = parse_bool(frame["pred_interval"])
    frame["prob_interval"] = pd.to_numeric(frame["prob_interval"], errors="coerce")
    frame["net_load_MW"] = pd.to_numeric(frame["net_load_MW"], errors="coerce")
    frame["solar_MW"] = pd.to_numeric(frame["solar_MW"], errors="coerce")
    frame["hour"] = frame["timestamp_dt"].dt.hour
    frame["month"] = frame["timestamp_dt"].dt.month
    return frame


def confusion_label(y_true: pd.Series, y_pred: pd.Series) -> pd.Series:
    true = y_true.astype(bool)
    pred = y_pred.astype(bool)
    return pd.Series(
        np.select(
            [true & pred, ~true & pred, true & ~pred, ~true & ~pred],
            ["TP", "FP", "FN", "TN"],
            default="?",
        ),
        index=y_true.index,
    )


def binary_metric_row(y_true: Iterable[bool], y_pred: Iterable[bool]) -> dict[str, float | int]:
    true = np.asarray(list(y_true), dtype=bool)
    pred = np.asarray(list(y_pred), dtype=bool)
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "support": int(len(true)),
        "positive_support": int(true.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def build_day_diagnostics(interval_df: pd.DataFrame) -> pd.DataFrame:
    base = interval_df.groupby(["dataset", "fold_id", "substation_id", "date"], as_index=False).agg(
        label_day=("label_interval", "any"),
        final_pred_day=("pred_interval", "any"),
        label_intervals=("label_interval", "sum"),
        pred_intervals=("pred_interval", "sum"),
        xgb1_candidate_intervals=("prob_interval", lambda s: int(s.notna().sum())),
        max_prob_interval=("prob_interval", "max"),
        mean_prob_interval=("prob_interval", "mean"),
        solar_sum=("solar_MW", "sum"),
        solar_max=("solar_MW", "max"),
        net_min=("net_load_MW", "min"),
        net_max=("net_load_MW", "max"),
        net_missing=("net_load_MW", lambda s: int(s.isna().sum())),
        solar_missing=("solar_MW", lambda s: int(s.isna().sum())),
        n_rows=("timestamp", "size"),
    )
    daytime = interval_df.loc[interval_df["hour"].between(6, 18, inclusive="both")]
    noon = daytime.groupby(["dataset", "fold_id", "substation_id", "date"], as_index=False).agg(
        solar_daytime_max=("solar_MW", "max"),
        net_noon_min=("net_load_MW", "min"),
        net_noon_max=("net_load_MW", "max"),
        noon_missing=("net_load_MW", lambda s: int(s.isna().sum())),
        noon_rows=("timestamp", "size"),
    )
    day = base.merge(noon, on=["dataset", "fold_id", "substation_id", "date"], how="left")
    day["xgb1_candidate_day"] = day["xgb1_candidate_intervals"] > 0
    day["xgb1_confusion"] = confusion_label(day["label_day"], day["xgb1_candidate_day"])
    day["final_confusion"] = confusion_label(day["label_day"], day["final_pred_day"])
    day["month"] = pd.to_datetime(day["date"]).dt.month
    return day.sort_values(["dataset", "substation_id", "date"]).reset_index(drop=True)


def metric_rows(day_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for scope, group_cols in [
        ("dataset", ["dataset"]),
        ("site", ["dataset", "substation_id"]),
    ]:
        for keys, group in day_df.groupby(group_cols, dropna=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            key_payload = dict(zip(group_cols, keys))
            for decision, pred_col in [
                ("xgb1_candidate_day", "xgb1_candidate_day"),
                ("final_after_xgb2", "final_pred_day"),
            ]:
                rows.append(
                    {
                        "summary_scope": scope,
                        **key_payload,
                        "decision": decision,
                        **binary_metric_row(group["label_day"], group[pred_col]),
                    }
                )
    return pd.DataFrame(rows)


def beta_site_month_breakdown(day_df: pd.DataFrame) -> pd.DataFrame:
    beta = day_df.loc[day_df["dataset"].eq("Beta")].copy()
    grouped = (
        beta.groupby(["substation_id", "month", "xgb1_confusion"], as_index=False)
        .size()
        .rename(columns={"size": "n_days"})
    )
    return grouped.sort_values(["substation_id", "month", "xgb1_confusion"]).reset_index(drop=True)


def beta_feature_summary(day_df: pd.DataFrame) -> pd.DataFrame:
    beta = day_df.loc[day_df["dataset"].eq("Beta")].copy()
    feature_cols = [
        "solar_daytime_max",
        "solar_sum",
        "net_noon_min",
        "net_noon_max",
        "label_intervals",
        "pred_intervals",
        "xgb1_candidate_intervals",
        "net_missing",
        "solar_missing",
        "noon_missing",
    ]
    pieces = []
    for confusion_col in ["xgb1_confusion", "final_confusion"]:
        summary = beta.groupby(confusion_col)[feature_cols].agg(["count", "median", "mean", "min", "max"])
        summary.columns = [f"{feature}_{stat}" for feature, stat in summary.columns]
        summary = summary.reset_index().rename(columns={confusion_col: "confusion"})
        summary.insert(0, "confusion_basis", confusion_col)
        pieces.append(summary)
    return pd.concat(pieces, ignore_index=True, sort=False)


def select_informative_examples(day_df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    beta = day_df.loc[day_df["dataset"].eq("Beta")].copy()
    beta["low_noon_net_score"] = -beta["net_noon_min"].fillna(beta["net_noon_min"].max())
    beta["solar_score"] = beta["solar_daytime_max"].fillna(0)
    beta["interval_score"] = beta[["label_intervals", "pred_intervals"]].max(axis=1).fillna(0)
    beta["missing_score"] = beta[["net_missing", "solar_missing"]].sum(axis=1)
    rows = []
    for (site, confusion), group in beta.groupby(["substation_id", "xgb1_confusion"]):
        ranked = group.sort_values(
            ["interval_score", "solar_score", "low_noon_net_score", "missing_score", "date"],
            ascending=[False, False, False, False, True],
        ).head(n)
        rows.append(ranked)
    if not rows:
        return pd.DataFrame(columns=beta.columns)
    examples = pd.concat(rows, ignore_index=True)
    keep_cols = [
        "substation_id",
        "date",
        "xgb1_confusion",
        "final_confusion",
        "label_day",
        "xgb1_candidate_day",
        "final_pred_day",
        "label_intervals",
        "pred_intervals",
        "xgb1_candidate_intervals",
        "solar_daytime_max",
        "solar_sum",
        "net_noon_min",
        "net_noon_max",
        "net_missing",
        "solar_missing",
        "month",
    ]
    return examples[keep_cols].sort_values(["substation_id", "xgb1_confusion", "date"]).reset_index(drop=True)


## 3. Load Prediction CSVs And Build Day-Level Diagnostics

This cell reads the real `m8_xgb` prediction CSVs, builds interval-level and day-level diagnostic frames, and writes the main diagnostic CSVs. The output tables intentionally include both XGB1-candidate and final-after-XGB2 decisions.


In [ ]:
alpha_frames = []
for label, path in required_prediction_files.items():
    if label.startswith("Alpha"):
        site = label.split()[-1]
        alpha_frames.append(load_prediction_csv(path, "Alpha", f"alpha_holdout_{site}"))

beta_interval = load_prediction_csv(
    required_prediction_files["Beta transfer"], "Beta", "beta_transfer"
)
alpha_interval = pd.concat(alpha_frames, ignore_index=True)
interval_predictions = pd.concat([alpha_interval, beta_interval], ignore_index=True)
day_diagnostics = build_day_diagnostics(interval_predictions)
metrics = metric_rows(day_diagnostics)
beta_breakdown = beta_site_month_breakdown(day_diagnostics)
feature_summary = beta_feature_summary(day_diagnostics)
selected_examples = select_informative_examples(day_diagnostics, n=10)

metrics.to_csv(csv_dir / "01_day_candidate_metrics.csv", index=False)
beta_breakdown.to_csv(csv_dir / "02_beta_xgb1_confusion_by_site_month.csv", index=False)
feature_summary.to_csv(csv_dir / "03_beta_feature_summary_by_confusion.csv", index=False)
selected_examples.to_csv(csv_dir / "04_beta_selected_examples.csv", index=False)
day_diagnostics.to_csv(csv_dir / "05_day_level_diagnostics.csv", index=False)

print(f"Interval rows: {len(interval_predictions):,}")
print(f"Day rows: {len(day_diagnostics):,}")
print(f"Selected example rows: {len(selected_examples):,}")
display(metrics.sort_values(["summary_scope", "dataset", "substation_id", "decision"]).head(30))


## 4. Static Diagnostic Figures

These figures provide a quick high-level read before opening the Plotly examples. They focus on the XGB1 candidate step because that is where the Alpha-to-Beta transfer drop first appears.


In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "axes.edgecolor": PALETTE["dark_blue"],
    "axes.labelcolor": PALETTE["dark_blue"],
    "axes.titlecolor": PALETTE["dark_blue"],
    "xtick.color": PALETTE["dark_blue"],
    "ytick.color": PALETTE["dark_blue"],
    "text.color": PALETTE["dark_blue"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# Figure 1: Alpha vs Beta aggregate score comparison.
agg = metrics.loc[metrics["summary_scope"].eq("dataset")].copy()
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), sharey=True)
for ax, decision, title in zip(
    axes,
    ["xgb1_candidate_day", "final_after_xgb2"],
    ["XGB1 candidate day", "Final day after XGB2"],
):
    plot = agg.loc[agg["decision"].eq(decision)].set_index("dataset").reindex(["Alpha", "Beta"])
    x = np.arange(len(plot.index))
    width = 0.22
    for idx, metric in enumerate(SCORE_COLUMNS):
        values = plot[metric].to_numpy(dtype=float)
        bars = ax.bar(
            x + (idx - 1) * width,
            values,
            width,
            label=metric.upper() if metric == "f1" else metric.title(),
            color=[PALETTE["dark_blue"], PALETTE["orange"], PALETTE["grey"]][idx],
        )
        ax.bar_label(bars, labels=[f"{value:.2f}" for value in values], fontsize=8, padding=2)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(plot.index)
    ax.set_ylim(0, 1.08)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], alpha=0.8)
axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Alpha vs Beta day-level m8_xgb diagnostic scores", y=0.99)
fig.subplots_adjust(bottom=0.22, top=0.80, wspace=0.10)
fig.savefig(fig_dir / "fig01_alpha_beta_xgb1_vs_final_scores.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Figure 2: Beta site-level XGB1 candidate scores.
site_scores = metrics.loc[
    metrics["summary_scope"].eq("site")
    & metrics["dataset"].eq("Beta")
    & metrics["decision"].eq("xgb1_candidate_day")
].copy()
site_scores["site_order"] = site_scores["substation_id"].map(
    {site: idx for idx, site in enumerate(PRIORITY_BETA_SITES)}
).fillna(99)
site_scores = site_scores.sort_values(["site_order", "f1", "substation_id"], ascending=[True, True, True])
fig, ax = plt.subplots(figsize=(8.8, 4.2))
x = np.arange(len(site_scores))
width = 0.24
for idx, metric in enumerate(SCORE_COLUMNS):
    values = site_scores[metric].to_numpy(dtype=float)
    bars = ax.bar(
        x + (idx - 1) * width,
        values,
        width,
        label=metric.upper() if metric == "f1" else metric.title(),
        color=[PALETTE["dark_blue"], PALETTE["orange"], PALETTE["grey"]][idx],
    )
    ax.bar_label(bars, labels=[f"{value:.2f}" for value in values], fontsize=7, padding=2)
ax.set_title("Beta site-level XGB1 candidate-day scores")
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels(site_scores["substation_id"], rotation=35, ha="right")
ax.set_ylim(0, 1.08)
ax.legend(ncol=3, frameon=False)
ax.set_axisbelow(True)
ax.grid(axis="y", color=PALETTE["light_white"], alpha=0.8)
fig.tight_layout()
fig.savefig(fig_dir / "fig02_beta_site_xgb1_scores.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Figure 3: Beta feature distributions by XGB1 confusion group.
beta_day = day_diagnostics.loc[day_diagnostics["dataset"].eq("Beta")].copy()
features = [
    ("solar_daytime_max", "Daytime solar peak (MW)"),
    ("net_noon_min", "Daytime minimum net load (MW)"),
    ("label_intervals", "Labelled RPF intervals"),
    ("net_missing", "Missing net-load intervals"),
]
fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.4))
for ax, (feature, label) in zip(axes.ravel(), features):
    data = [
        beta_day.loc[beta_day["xgb1_confusion"].eq(group), feature].dropna().to_numpy()
        for group in CONFUSION_ORDER
    ]
    box = ax.boxplot(data, tick_labels=CONFUSION_ORDER, patch_artist=True, showfliers=False)
    for idx, patch in enumerate(box["boxes"]):
        patch.set_facecolor([PALETTE["dark_blue"], PALETTE["orange"], PALETTE["grey"], PALETTE["light_grey"]][idx])
        patch.set_alpha(0.72)
        patch.set_edgecolor(PALETTE["dark_blue"])
    for element in ["whiskers", "caps", "medians"]:
        for item in box[element]:
            item.set_color(PALETTE["dark_blue"])
    ax.set_title(label)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], alpha=0.8)
fig.suptitle("Beta XGB1 candidate-day feature distributions by confusion group", y=0.99)
fig.tight_layout()
fig.savefig(fig_dir / "fig03_beta_xgb1_feature_distributions.png", dpi=220, bbox_inches="tight")
plt.close(fig)

print("Wrote static figures to", fig_dir)


## 5. Plotly Example Browser Outputs

This section writes interactive HTML files for up to ten deterministic ?most informative? examples per Beta site and XGB1 confusion group. Open the HTML files directly, or use `show_examples(site, confusion)` in the next section for notebook exploration.


In [ ]:
def contiguous_spans(mask: pd.Series, timestamps: pd.Series) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    spans = []
    in_span = False
    start = None
    last = None
    for flag, ts in zip(mask.astype(bool).to_numpy(), timestamps):
        if flag and not in_span:
            start = ts
            in_span = True
        if in_span and (not flag):
            spans.append((start, last + pd.Timedelta(minutes=15)))
            in_span = False
        if flag:
            last = ts
    if in_span and start is not None and last is not None:
        spans.append((start, last + pd.Timedelta(minutes=15)))
    return spans


def plot_example_group(
    interval_df: pd.DataFrame,
    examples: pd.DataFrame,
    site: str,
    confusion: str,
    max_examples: int = 10,
) -> go.Figure | None:
    chosen = examples.loc[
        examples["substation_id"].eq(site) & examples["xgb1_confusion"].eq(confusion)
    ].head(max_examples)
    if chosen.empty:
        return None
    rows = len(chosen)
    fig = make_subplots(
        rows=rows,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.035,
        specs=[[{"secondary_y": True}] for _ in range(rows)],
        subplot_titles=[
            f"{row.date} | final={row.final_confusion} | label_int={row.label_intervals} | pred_int={row.pred_intervals} | noon_min={row.net_noon_min:.3g} MW | solar_peak={row.solar_daytime_max:.3g} MW"
            for row in chosen.itertuples(index=False)
        ],
    )
    for row_idx, row in enumerate(chosen.itertuples(index=False), start=1):
        day = interval_df.loc[
            interval_df["substation_id"].eq(site) & interval_df["date"].eq(row.date)
        ].sort_values("timestamp_dt")
        fig.add_trace(
            go.Scatter(
                x=day["timestamp_dt"],
                y=day["net_load_MW"],
                mode="lines",
                name="Raw net load",
                line=dict(color=PALETTE["dark_blue"], width=1.8),
                showlegend=row_idx == 1,
            ),
            row=row_idx,
            col=1,
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=day["timestamp_dt"],
                y=day["solar_MW"],
                mode="lines",
                name="Solar generation",
                line=dict(color=PALETTE["orange"], width=1.5),
                showlegend=row_idx == 1,
            ),
            row=row_idx,
            col=1,
            secondary_y=True,
        )
        for start, end in contiguous_spans(day["label_interval"], day["timestamp_dt"]):
            fig.add_vrect(
                x0=start,
                x1=end,
                fillcolor=PALETTE["orange"],
                opacity=0.18,
                line_width=0,
                row=row_idx,
                col=1,
            )
        for start, end in contiguous_spans(day["pred_interval"], day["timestamp_dt"]):
            fig.add_vrect(
                x0=start,
                x1=end,
                fillcolor=PALETTE["dark_blue"],
                opacity=0.16,
                line_width=0,
                row=row_idx,
                col=1,
            )
        fig.add_hline(y=0, line=dict(color=PALETTE["grey"], width=1, dash="dot"), row=row_idx, col=1)
        fig.update_yaxes(title_text="Net load (MW)", row=row_idx, col=1, secondary_y=False)
        fig.update_yaxes(title_text="Solar (MW)", row=row_idx, col=1, secondary_y=True)
    fig.update_layout(
        title=f"{site} XGB1 {confusion} examples (orange shade=label, blue shade=final predicted interval)",
        height=max(320, 230 * rows),
        width=1150,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1),
    )
    return fig

html_index_rows = []
for site in sorted(selected_examples["substation_id"].unique()):
    for confusion in CONFUSION_ORDER:
        fig = plot_example_group(beta_interval, selected_examples, site, confusion, max_examples=10)
        if fig is None:
            continue
        out_path = html_dir / f"{site}_{confusion}_examples.html"
        fig.write_html(out_path, include_plotlyjs="cdn", full_html=True)
        html_index_rows.append(
            {
                "substation_id": site,
                "xgb1_confusion": confusion,
                "n_examples": int(
                    len(
                        selected_examples.loc[
                            selected_examples["substation_id"].eq(site)
                            & selected_examples["xgb1_confusion"].eq(confusion)
                        ]
                    )
                ),
                "html_file": str(out_path.relative_to(output_dir)),
            }
        )
html_index = pd.DataFrame(html_index_rows)
html_index.to_csv(csv_dir / "06_plotly_html_index.csv", index=False)
print(f"Wrote {len(html_index)} Plotly HTML files to {html_dir}")
display(html_index.head(20))


## 6. Interactive Notebook Exploration

Use the variables below to inspect one site/confusion group inside the notebook. The saved HTML files from the previous section are usually faster for broad browsing, while this function is useful for ad hoc checks.


In [ ]:
# Change these values and rerun this cell for ad hoc notebook exploration.
SELECTED_SITE = "beta_B"
SELECTED_CONFUSION = "FN"  # TP, FP, FN, or TN
N_EXAMPLES = 10

fig = plot_example_group(
    beta_interval,
    selected_examples,
    SELECTED_SITE,
    SELECTED_CONFUSION,
    max_examples=N_EXAMPLES,
)
if fig is None:
    print(f"No examples found for {SELECTED_SITE} / {SELECTED_CONFUSION}")
else:
    fig.show()


## 7. Quick Tables To Inspect

These previews are intentionally small. For full detail, open the CSV files in `notebooks/99_Misc/outputs/01_beta_xgb1_diagnostics/csv/`.


In [ ]:
print("Dataset-level metrics")
display(metrics.loc[metrics["summary_scope"].eq("dataset")])

print("Beta site-level XGB1 candidate metrics")
display(
    metrics.loc[
        metrics["summary_scope"].eq("site")
        & metrics["dataset"].eq("Beta")
        & metrics["decision"].eq("xgb1_candidate_day")
    ].sort_values("f1")
)

print("Beta selected examples")
display(selected_examples.head(30))
